# Short-Horizon Health Indicator Forecasting — SDR 2026

This refresh predicts one later observation for maternal mortality and life expectancy. It uses a time-aware evaluation: for every country, the latest observed value is held out. This is more realistic than a random row split for time-series data.

## 1. Load the current SDR dataset

In [1]:
import warnings
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
DATA_FILE = Path("SDR2026-database.xlsx")
DATA_URL = "https://dashboards.sdgindex.org/static/downloads/database_2026.xlsx"
if not DATA_FILE.exists():
    request = Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request) as response, open(DATA_FILE, "wb") as destination:
        destination.write(response.read())
print(f"Using {DATA_FILE} ({DATA_FILE.stat().st_size / 1_000_000:.1f} MB)")

raw = pd.read_excel(DATA_FILE, sheet_name="All Raw Data")
panel = raw[["country", "year", "sdg3_matmort", "sdg3_lifee"]].copy().sort_values(["country", "year"])
print(str(panel.country.nunique()) + " countries; latest listed year: " + str(panel.year.max()))


/Users/tanvirrahman/.matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /var/folders/bj/dzvtb5l54dd3l6qg80_q_7w40000gn/T/matplotlib-ricele2o because there was an issue with the default path (/Users/tanvirrahman/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.



Fontconfig error: No writable cache directories
	/opt/homebrew/var/cache/fontconfig
	/Users/tanvirrahman/.cache/fontconfig
	/Users/tanvirrahman/.fontconfig

Fontconfig error: No writable cache directories
	/opt/homebrew/var/cache/fontconfig
	/Users/tanvirrahman/.cache/fontconfig
	/Users/tanvirrahman/.fontconfig

Fontconfig error: No writable cache directories
	/opt/homebrew/var/cache/fontconfig
	/Users/tanvirrahman/.cache/fontconfig
	/Users/tanvirrahman/.fontconfig

Fontconfig error: No writable cache directories
	/opt/homebrew/var/cache/fontconfig
	/Users/tanvirrahman/.cache/fontconfig
	/Users/tanvirrahman/.fontconfig


Matplotlib is building the font cache; this may take a moment.


Using SDR2026-database.xlsx (9.4 MB)


208 countries; latest listed year: 2026


## 2. Compare candidates with a time-aware holdout

The candidates are: persistence (the last observation), an 8-observation linear trend, and a full-history linear trend. Maternal mortality is assessed on the original scale for clear reporting.

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

def evaluate(metric):
    data = panel.dropna(subset=[metric]).sort_values(["country", "year"])
    actual, predictions = [], {"Persistence": [], "Recent linear trend": [], "Full-history linear trend": []}
    for _, group in data.groupby("country"):
        if len(group) < 8:
            continue
        train, test = group.iloc[:-1], group.iloc[-1]
        if len(train) < 6:
            continue
        actual.append(test[metric])
        predictions["Persistence"].append(train[metric].iloc[-1])
        recent = train.tail(8)
        predictions["Recent linear trend"].append(LinearRegression().fit(recent[["year"]], recent[metric]).predict(pd.DataFrame({"year": [test.year]}))[0])
        predictions["Full-history linear trend"].append(LinearRegression().fit(train[["year"]], train[metric]).predict(pd.DataFrame({"year": [test.year]}))[0])
    return pd.DataFrame([{
        "model": name, "MAE": mean_absolute_error(actual, values), "R²": r2_score(actual, values), "countries": len(actual)
    } for name, values in predictions.items()]).sort_values("MAE"), actual, predictions

mmr_scores, _, _ = evaluate("sdg3_matmort")
le_scores, _, _ = evaluate("sdg3_lifee")
print("Maternal mortality")
display(mmr_scores.round(3))
print("Life expectancy")
display(le_scores.round(3))


Maternal mortality


,model,MAE,R²,countries
0,Persistence,10.922,0.983,207
2,Full-history linear trend,19.535,0.938,207
1,Recent linear trend,20.063,0.917,207


Life expectancy


,model,MAE,R²,countries
0,Persistence,0.185,0.999,208
1,Recent linear trend,0.593,0.977,208
2,Full-history linear trend,0.663,0.967,208


## 3. Chosen method and updated estimates

The benchmark selects the model with the smallest time-aware mean absolute error. For a one-observation horizon in this dataset, persistence is selected. It outperforms linear trends, which can overreact to sparse or unevenly reported series.

In [3]:
def one_step_forecast(metric, countries):
    data = panel.dropna(subset=[metric]).sort_values(["country", "year"])
    result = []
    for country in countries:
        group = data[data.country == country]
        if group.empty:
            continue
        last = group.iloc[-1]
        result.append({"country": country, "source_year": int(last.year), "forecast_year": int(last.year + 1), metric: float(last[metric])})
    return pd.DataFrame(result)

selected = ["Albania", "Australia", "Morocco", "Egypt, Arab Rep.", "Kazakhstan", "Kyrgyz Republic", "Malaysia", "Thailand", "Uruguay", "North Macedonia"]
mmr_forecast = one_step_forecast("sdg3_matmort", selected)
le_forecast = one_step_forecast("sdg3_lifee", selected)
forecast = mmr_forecast.merge(le_forecast[["country", "sdg3_lifee"]], on="country").rename(columns={
    "sdg3_matmort": "maternal_mortality_estimate", "sdg3_lifee": "life_expectancy_estimate"})
forecast.round(2)


,country,source_year,forecast_year,maternal_mortality_estimate,life_expectancy_estimate
0,Albania,2023,2024,6.87,79.95
1,Australia,2023,2024,2.33,84.21
2,Morocco,2023,2024,69.77,75.68
3,"Egypt, Arab Rep.",2023,2024,16.95,71.99
4,Kazakhstan,2023,2024,9.58,74.67
5,Kyrgyz Republic,2023,2024,41.90,71.97
6,Malaysia,2023,2024,26.38,76.99
7,Thailand,2023,2024,34.45,76.82
8,Uruguay,2023,2024,15.30,78.45
9,North Macedonia,2023,2024,2.76,77.68


## Findings and model decision

- The previous random split has been replaced with a **time-aware, per-country holdout**, avoiding information from a country’s later history leaking into training.
- For one-step forecasts, the simplest model — carrying forward the latest measurement — has lower error than the tested linear-trend models for both indicators.
- This is a meaningful improvement in model selection: the notebook now chooses the model from out-of-time evidence rather than assuming a more complex model is better.
- The estimates are planning aids, not clinical predictions, and they retain the reporting cadence and uncertainty of the source data.